# Incremental impact explorer (SageMaker Unified Studio)

Interactive companion to the scheduled `wm_incremental_impact` Glue job. It reads the gold
campaign panel through Athena (Lake Formation permissions apply), re-runs the
difference-in-differences estimator with any settings you choose, and plots treatment
against control.

**Prerequisites:** a Unified Studio project with access to the `wm_gold` and `wm_semantic`
Glue databases, and this repository cloned into the project (Git connection).

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../src"))           # repo cloned into the project

import awswrangler as wr
import matplotlib.pyplot as plt
import pandas as pd

from wm_common.impact import METRIC_REGISTRY, did_estimate, evaluate_campaign

WORKGROUP = "wm-dev-analytics"                         # Athena workgroup from Terraform
pd.set_option("display.width", 200)

## 1 · Load the campaign panel (household × month, treatment vs control, pre vs post)

In [ ]:
panel = wr.athena.read_sql_query(
    "SELECT * FROM wm_gold.fct_campaign_household_monthly",
    database="wm_gold", workgroup=WORKGROUP, ctas_approach=False)
panel.groupby(["campaign_id", "assignment_group", "period"]).household_id.nunique().unstack()

## 2 · Is the randomization balanced before launch?

In [ ]:
pre = panel[panel.period == "pre"]
pre.groupby(["campaign_id", "assignment_group"])[["net_flows_cad", "interactions", "digital_sessions", "complaints"]].mean().round(2)

## 3 · Treatment vs control over time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (cid, metric) in zip(axes, [("CMP-2026-01", "net_flows_cad"), ("CMP-2026-02", "digital_sessions")]):
    trend = (panel[panel.campaign_id == cid]
             .groupby(["rel_month", "assignment_group"])[metric].mean().unstack())
    trend.plot(ax=ax, marker="o", title=f"{cid} · {metric}")
    ax.axvline(-0.5, ls="--", c="grey"); ax.set_xlabel("months from launch")
plt.tight_layout()

## 4 · Difference-in-differences with bootstrap CIs (quantity + quality guardrails)

In [ ]:
results = pd.concat([
    evaluate_campaign(g, cid, [m for m in METRIC_REGISTRY if m in g.columns], n_boot=1000)
    for cid, g in panel.groupby("campaign_id")])
results[["campaign_id", "metric", "dimension", "did_estimate", "ci_low", "ci_high",
         "relative_lift", "pre_trend_ok", "verdict"]].round(3)

## 5 · Sensitivity: does the answer change with a shorter post window?

In [ ]:
cid, metric = "CMP-2026-01", "net_flows_cad"
p = panel[panel.campaign_id == cid]
rows = []
for last in sorted(p.loc[p.period == "post", "rel_month"].unique()):
    r = did_estimate(p[(p.period == "pre") | (p.rel_month <= last)], metric, cid, n_boot=500)
    rows.append({"post_months": last + 1, "estimate": r.did_estimate, "ci_low": r.ci_low, "ci_high": r.ci_high})
pd.DataFrame(rows).round(0)

## 6 · (Optional) log to SageMaker managed MLflow

In [ ]:
# import mlflow
# mlflow.set_tracking_uri("arn:aws:sagemaker:<region>:<account>:mlflow-tracking-server/wm-analytics")
# mlflow.set_experiment("wm-incremental-impact")
# with mlflow.start_run(run_name="explorer"):
#     for r in results.itertuples():
#         mlflow.log_metric(f"{r.campaign_id}_{r.metric}_did", r.did_estimate)